# Preprocessing & Model Training - RT-IoT2022

# Problem Formalization

Task: IoT Intrusion Detection System

Type: Supervised Multi-Class Classification

Goal: Predict the type of network traffic (attack types or normal traffic) based on IoT network features

Approach:
1. Preprocess data (encoding, normalization)
2. Train/test split
3. Train baseline model (Logistic Regression)
4. Evaluate performance

In [ ]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
# importation take too much time so i moved it down here
from ucimlrepo import fetch_ucirepo

rt_iot2022 = fetch_ucirepo(id=942)
X = rt_iot2022.data.features
y = rt_iot2022.data.targets

## Data Preprocessing

In [14]:
for col in X.select_dtypes(include=['object']).columns:
    X[col] = LabelEncoder().fit_transform(X[col])

X_scaled = StandardScaler().fit_transform(X)

y_encoded = LabelEncoder().fit_transform(y.iloc[:, 0])

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.3, random_state=20, stratify=y_encoded
)

print("Train: ",X_train.shape, "\nTest : ", X_test.shape)

Train: (86181, 83), Test: (36936, 83)


## Model Training & Evaluation

In [21]:
model = LogisticRegression(max_iter=200, random_state=20)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification report:\n", classification_report(y_test, y_pred))

Accuracy: 0.9900638943036604

Classification report:
               precision    recall  f1-score   support

           0       0.95      0.92      0.93      2325
           1       0.96      0.78      0.86       160
           2       1.00      1.00      1.00     28398
           3       1.00      1.00      1.00      1244
           4       1.00      0.82      0.90        11
           5       0.78      0.88      0.82         8
           6       0.99      1.00      1.00       600
           7       0.99      1.00      0.99       301
           8       0.95      0.98      0.97       777
           9       1.00      1.00      1.00       603
          10       0.93      0.96      0.94      2433
          11       0.80      0.62      0.70        76

    accuracy                           0.99     36936
   macro avg       0.95      0.91      0.93     36936
weighted avg       0.99      0.99      0.99     36936



C:\Users\ender\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 200 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=200).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Correct Predictions by Attack Type

In [ ]:
attack = le_target.inverse_transform(y_test)
predictions = le_target.inverse_transform(y_pred)

results_df = pd.DataFrame({
    'Actual': attack,
    'Predicted': predictions,
    'Correct': attack == predictions
})

correct = results_df[results_df['Correct'] == True].groupby('Actual').size()
total = results_df.groupby('Actual').size()

summary = pd.DataFrame({
    'Total': total,
    'Correct': correct,
    'Accuracy': (correct / total * 100)
})

print("Correct predictions by attack type:\n")
print(summary.sort_values('Accuracy', ascending=False))

Correct predictions by attack type:

                            Total  Correct  Accuracy
Actual                                              
DOS_SYN_Hping               28398    28398    100.00
NMAP_TCP_scan                 301      301    100.00
NMAP_OS_DETECTION             600      600    100.00
MQTT_Publish                 1244     1241     99.76
NMAP_XMAS_TREE_SCAN           603      601     99.67
NMAP_UDP_SCAN                 777      761     97.94
Thing_Speak                  2433     2338     96.10
ARP_poisioning               2325     2142     92.13
NMAP_FIN_SCAN                   8        7     87.50
Metasploit_Brute_Force_SSH     11        9     81.82
DDOS_Slowloris                160      124     77.50
Wipro_bulb                     76       47     61.84
